In [ ]:
from dask.distributed import Client

client = Client()

In [ ]:
client.close()

In [9]:
import ee
import geemap
import xarray

ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com', project="calfuels")

def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)

    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
    'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)


CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

ic = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())
      .filterDate('2019-01-01', '2019-12-31').select("SR_B.|SR_B7"))

In [10]:
import geemap

geemap.ee_export_image_collection_to_drive(ic, folder="going_down_on_me", scale=30, crs="EPSG:3310", region=LTBMU.geometry())

Total number of images: 22

